# Descargador de series BCRP — EF PUCP 2026-1, Pregunta 1

**Réplica de Lahura (2017), "El efecto traspaso de la tasa de interés de política monetaria en Perú: Evidencia reciente"**

Adaptado del scraper del proyecto La Serna (`bcrp_series.ipynb` original) para este examen final. Cambios respecto de la versión original:

- Se elimina el flujo de escritura a `la_serna_datos.xlsx` (multi-sheet + hoja `00_metadatos`), que no aplica aquí.
- Se pre-configuran directamente los 17 códigos de tasas bancarias del Cuadro 1 de Lahura (2017), más la tasa
  de referencia de política monetaria y la tasa interbancaria (para el Gráfico 4).
- La salida es un único panel mensual limpio (`tasas_interes_lahura2017.xlsx` / `.csv`), listo para importar en EViews 12.
- Se conserva el principio de descarga del original: **una serie a la vez** vía `bcrp_una_serie_mensual`,
  porque el API multi-serie del BCRP no garantiza el orden de los valores devueltos (bug ya documentado
  en el notebook original del proyecto La Serna).

**Muestra**: agosto 2010 – mayo 2017, igual que en el paper original (Lahura, 2017, p. 11).

In [1]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import requests

# ── Rutas ─────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.path.abspath(''))          # ef_pucp/data
OUT_XLSX     = NOTEBOOK_DIR / 'tasas_interes_lahura2017.xlsx'
OUT_CSV      = NOTEBOOK_DIR / 'tasas_interes_lahura2017.csv'

# Detectar el CSV de metadatos mas reciente en la carpeta (heredado del scraper original;
# no es necesario para la descarga, solo para verificar codigos manualmente)
csvs = sorted(glob.glob(str(NOTEBOOK_DIR / 'BCRPData-metadata-*.csv')))
METADATA_CSV = Path(csvs[-1]) if csvs else None

BASE_URL = 'https://estadisticas.bcrp.gob.pe/estadisticas/series/api'

print('✓ Librerías cargadas.')
print(f'  Carpeta datos : {NOTEBOOK_DIR}')
print(f'  Metadata      : {METADATA_CSV.name if METADATA_CSV else "no encontrado"}')

✓ Librerías cargadas.
  Carpeta datos : /ruta/ef_pucp/data
  Metadata      : BCRPData-metadata-20260707-125357.csv


In [2]:
# ══ FUNCIÓN: bcrp_una_serie_mensual ═══════════════════════════════════════
# Descarga UNA serie mensual del BCRP vía API pública (misma logica de
# seguridad que bcrp_mensual del notebook original: nunca pedir varias
# series en una sola URL, para evitar el bug de orden del API del BCRP).

_MESES = {'Ene':1,'Feb':2,'Mar':3,'Abr':4,'May':5,'Jun':6,
          'Jul':7,'Ago':8,'Set':9,'Sep':9,'Oct':10,'Nov':11,'Dic':12}

def bcrp_una_serie_mensual(codigo, inicio, fin):
    """
    Descarga UNA serie mensual del BCRP vía API pública.
    inicio, fin: strings tipo 'YYYY-M' (ej. '2010-8', '2017-5').
    Retorna pd.Series indexada por Timestamp (primer dia del mes).
    """
    url  = f"{BASE_URL}/{codigo}/json/{inicio}/{fin}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if not data.get('periods'):
        raise ValueError(f'API sin datos para {codigo}. URL: {url}')
    fechas, valores = [], []
    for p in data['periods']:
        mes_str, yr_str = p['name'].split('.')
        mes = _MESES[mes_str]
        yr  = int(yr_str)
        fechas.append(pd.Timestamp(year=yr, month=mes, day=1))
        try:
            valores.append(float(p['values'][0]))
        except (ValueError, TypeError):
            valores.append(np.nan)
    return pd.Series(valores, index=pd.DatetimeIndex(fechas, name='fecha'), name=codigo)


def bcrp_panel_mensual(codigos_dict, inicio, fin):
    """
    Descarga varias series mensuales (una por una) y las junta en un solo panel.
    codigos_dict: {nombre_variable: codigo_bcrp}
    """
    series = {}
    for nombre, codigo in codigos_dict.items():
        s = bcrp_una_serie_mensual(codigo, inicio, fin)
        series[nombre] = s
        print(f'  ✓ {nombre:10s} ({codigo}) — {len(s)} obs, {s.index.min():%Y-%m} a {s.index.max():%Y-%m}')
    return pd.DataFrame(series)

print('✓ Funciones definidas.')

✓ Funciones definidas.


## Series a descargar

Códigos verificados manualmente contra `BCRPData-metadata-*.csv` **y** contra el API en vivo. El campo
"Fecha de inicio" del CSV de metadatos está desactualizado para varios códigos (indica Ene-2011); el API
sí devuelve datos completos desde **Ago-2010**, que es el inicio real de la muestra usada por Lahura
(2017, p. 11): *"las estadísticas de las tasas de interés activas y pasivas... se encuentran disponibles
de manera continua desde agosto 2010."*

Nombres de variable alineados 1 a 1 con el Cuadro 1 del paper (tasas activas $R_1$–$R_9$, tasas pasivas
$R_{10}$–$R_{17}$), más $R_P$ (tasa de referencia, ecuación 1 del paper) y la tasa interbancaria
(solo para el Gráfico 4).

In [3]:
# ── Diccionario de series: {nombre_variable: codigo_BCRP} ─────────────────

CODIGOS = {
    # Tasas activas (R1-R9)
    'R1_pref90'   : 'PN07809NM',  # Preferencial Corporativa 90 dias
    'R2_corp_cp'  : 'PN07801NM',  # Corporativa hasta 360 dias
    'R3_ge_cp'    : 'PN07802NM',  # Grandes Empresas hasta 360 dias
    'R4_me_cp'    : 'PN07803NM',  # Medianas Empresas hasta 360 dias
    'R5_corp_lp'  : 'PN07804NM',  # Corporativa mayor a 360 dias
    'R6_ge_lp'    : 'PN07805NM',  # Grandes Empresas mayor a 360 dias
    'R7_me_lp'    : 'PN07806NM',  # Medianas Empresas mayor a 360 dias
    'R8_tamn'     : 'PN07807NM',  # TAMN
    'R9_ftamn'    : 'PN07808NM',  # FTAMN
    # Tasas pasivas (R10-R17)
    'R10_ctacte'  : 'PN07810NM',  # Cuenta Corriente
    'R11_ahorro'  : 'PN07811NM',  # Ahorros
    'R12_dp30'    : 'PN07812NM',  # Depositos hasta 30 dias
    'R13_dp180'   : 'PN07813NM',  # Depositos hasta 180 dias (BCRP: 31-180 dias)
    'R14_dp360'   : 'PN07814NM',  # Depositos hasta 360 dias (BCRP: 181-360 dias)
    'R15_dpmas360': 'PN07815NM',  # Depositos mayor a 360 dias
    'R16_tipmn'   : 'PN07816NM',  # TIPMN
    'R17_ftipmn'  : 'PN07817NM',  # FTIPMN
    # Tasa de politica monetaria y tasa interbancaria (para Grafico 4)
    'RP_ref'      : 'PD04722MM',  # Tasa de Referencia de la Politica Monetaria
    'R_interbank' : 'PN07819NM',  # Tasa Interbancaria Promedio
}

INICIO = '2010-8'   # Ago-2010, inicio real de la muestra de Lahura (2017)
FIN    = '2017-5'   # May-2017, fin de la muestra del paper

print(f'{len(CODIGOS)} series a descargar | muestra: {INICIO} -> {FIN}')

19 series a descargar | muestra: 2010-8 -> 2017-5


In [4]:
panel = bcrp_panel_mensual(CODIGOS, INICIO, FIN)
print(f'\nPanel final: {panel.shape[0]} obs x {panel.shape[1]} series')

  ✓ R1_pref90  (PN07809NM) — 82 obs, 2010-08 a 2017-05
  ✓ R2_corp_cp (PN07801NM) — 82 obs, 2010-08 a 2017-05
  ✓ R3_ge_cp   (PN07802NM) — 82 obs, 2010-08 a 2017-05
  ✓ R4_me_cp   (PN07803NM) — 82 obs, 2010-08 a 2017-05
  ✓ R5_corp_lp (PN07804NM) — 82 obs, 2010-08 a 2017-05
  ✓ R6_ge_lp   (PN07805NM) — 82 obs, 2010-08 a 2017-05
  ✓ R7_me_lp   (PN07806NM) — 82 obs, 2010-08 a 2017-05
  ✓ R8_tamn    (PN07807NM) — 82 obs, 2010-08 a 2017-05
  ✓ R9_ftamn   (PN07808NM) — 82 obs, 2010-08 a 2017-05
  ✓ R10_ctacte (PN07810NM) — 82 obs, 2010-08 a 2017-05
  ✓ R11_ahorro (PN07811NM) — 82 obs, 2010-08 a 2017-05
  ✓ R12_dp30   (PN07812NM) — 82 obs, 2010-08 a 2017-05
  ✓ R13_dp180  (PN07813NM) — 82 obs, 2010-08 a 2017-05
  ✓ R14_dp360  (PN07814NM) — 82 obs, 2010-08 a 2017-05
  ✓ R15_dpmas360 (PN07815NM) — 82 obs, 2010-08 a 2017-05
  ✓ R16_tipmn  (PN07816NM) — 82 obs, 2010-08 a 2017-05
  ✓ R17_ftipmn (PN07817NM) — 82 obs, 2010-08 a 2017-05
  ✓ RP_ref     (PD04722MM) — 82 obs, 2010-08 a 2017-05
  ✓ R_in

In [5]:
# ── Verificación de integridad ────────────────────────────────────────────
faltantes = panel.isna().sum()
faltantes = faltantes[faltantes > 0]
if faltantes.empty:
    print('✓ Sin valores faltantes en ninguna serie.')
else:
    print('⚠ Series con valores faltantes:')
    print(faltantes)

print(f'✓ {panel.shape[0]} observaciones mensuales verificadas (Ago.2010 - May.2017).')
print('Nota: el paper reporta 80 obs. efectivas en las pruebas de raiz unitaria y cointegracion')
print('tras la perdida de observaciones por rezagos/diferenciacion; la serie cruda del Cuadro 29')
print('tiene 82 meses (Ago.2010-May.2017), que es lo que se descarga aqui.')

✓ Sin valores faltantes en ninguna serie.
✓ 82 observaciones mensuales verificadas (Ago.2010 - May.2017).
Nota: el paper reporta 80 obs. efectivas en las pruebas de raiz unitaria y cointegracion
tras la perdida de observaciones por rezagos/diferenciacion; la serie cruda del Cuadro 29
tiene 82 meses (Ago.2010-May.2017), que es lo que se descarga aqui.


In [6]:
# ── Exportar (listo para importar en EViews 12) ───────────────────────────
panel_out = panel.reset_index()
panel_out['fecha'] = panel_out['fecha'].dt.strftime('%Ym%m')  # formato EViews: 2010m08

panel_out.to_csv(OUT_CSV, index=False)
panel_out.to_excel(OUT_XLSX, index=False, sheet_name='tasas_bcrp')

print(f'✓ Exportado: {OUT_CSV.name}  ({panel_out.shape[0]} filas x {panel_out.shape[1]} columnas)')
print(f'✓ Exportado: {OUT_XLSX.name}')

✓ Exportado: tasas_interes_lahura2017.csv  (82 filas x 20 columnas)
✓ Exportado: tasas_interes_lahura2017.xlsx


## Siguiente paso

En EViews 12: `File → Import → Import from File...`, seleccionar `tasas_interes_lahura2017.xlsx`,
frecuencia mensual, fecha de inicio `2010m08`. La columna `fecha` ya viene en formato `AAAAmMM`
(ej. `2010m08`), reconocible automáticamente por el wizard de importación de EViews.